# RT-DETR-L Training Notebook
Notebook thu nghiem backbone RT-DETR-L voi setting tuong tu YOLO de so sanh augmentation.

## 1. Thiet lap moi truong va thu vien
Cai dat/nhap cac thu vien can thiet cho RT-DETR-L va cac thu vien lien quan.

In [ ]:
!pip install pillow==10.2.0
# !pip install -U ultralytics huggingface_hub torch torchvision torchaudio opencv-python matplotlib tqdm pyyaml numpy
!pip install ultralytics huggingface_hub torch torchvision torchaudio opencv-python matplotlib tqdm pyyaml numpy

In [ ]:
# download dataset tu Google Drive
!pip install -q gdown

import gdown
import zipfile
import os

FILE_ID = "1gqjQld-jLWUjWuvCj6IuF0A3eKet5WwK"
ZIP_PATH = "/content/dataset.zip"
EXTRACT_PATH = "/content/dataset"

# download
gdown.download(f"https://drive.google.com/uc?id={FILE_ID}", ZIP_PATH, quiet=False)

# unzip
os.makedirs(EXTRACT_PATH, exist_ok=True)
with zipfile.ZipFile(ZIP_PATH, "r") as zip_ref:
    zip_ref.extractall(EXTRACT_PATH)

DATA_ROOT_DEFAULT = EXTRACT_PATH
print("Dataset ready:", EXTRACT_PATH)

In [ ]:
import os
import yaml
import random
import cv2
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch.utils.data import Dataset, DataLoader
from ultralytics import RTDETR

## 2. Cau hinh backbone RT-DETR-L voi setting tuong tu YOLO
Tao cau hinh hyperparameters de so sanh cong bang giua cac augmentation.

In [ ]:
# DATA_ROOT = "/content/drive/MyDrive/DAT/CAMO_Dataset_Full"
DATA_ROOT = "/content/dataset"
DATA_ROOT_SMM = "/content/dataset_smm"
USE_SMM = False

PROJECT_DIR = "DAT"
RUN_NAME = "rtdetr_l_camo_fs"
MODEL_NAME = "rtdetr-l.pt"

## 3. Nap du lieu va pipeline augmentation tuy chinh
Chon dataset (goc hoac da augment offline) va dinh nghia pipeline tuy chinh neu can.

In [ ]:
DATA_ROOT_ACTIVE = DATA_ROOT_SMM if USE_SMM else DATA_ROOT

assert os.path.exists(f"{DATA_ROOT_ACTIVE}/data.yaml"), "Khong thay data.yaml"
assert os.path.exists(f"{DATA_ROOT_ACTIVE}/train/images"), "Thieu train/images"
assert os.path.exists(f"{DATA_ROOT_ACTIVE}/train/labels"), "Thieu train/labels"
assert os.path.exists(f"{DATA_ROOT_ACTIVE}/val/images"), "Thieu val/images"
assert os.path.exists(f"{DATA_ROOT_ACTIVE}/val/labels"), "Thieu val/labels"

print("Dataset OK:", DATA_ROOT_ACTIVE)

yaml_path = f"{DATA_ROOT_ACTIVE}/data.yaml"
with open(yaml_path, "r") as f:
    data_yaml = yaml.safe_load(f)

data_yaml["path"] = DATA_ROOT_ACTIVE
data_yaml["train"] = "train/images"
data_yaml["val"] = "val/images"

with open(yaml_path, "w") as f:
    yaml.safe_dump(data_yaml, f, default_flow_style=False)

print("Da cap nhat data.yaml:", yaml_path)

# Pipeline augmentation tuy chinh (placeholder).
# Neu dung SMM offline, chi can set USE_SMM=True va data root tuong ung.
# Neu muon them mixup/cutmix tu code, ban se chen vao day.

def custom_augment(img):
    return img

## 4. DataLoader va kiem tra batch
Khoi tao DataLoader toi thieu va xem nhanh mot batch de kiem tra augmentation.

In [ ]:
class YoloFolderDataset(Dataset):
    def __init__(self, img_dir, augment=None, limit=None):
        self.img_paths = sorted([os.path.join(img_dir, p) for p in os.listdir(img_dir) if p.endswith(".jpg")])
        if limit:
            self.img_paths = self.img_paths[:limit]
        self.augment = augment

    def __len__(self):
        return len(self.img_paths)

    def __getitem__(self, idx):
        img_path = self.img_paths[idx]
        img = cv2.imread(img_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        if self.augment:
            img = self.augment(img)
        img = cv2.resize(img, (640, 640))
        img = img.astype(np.float32) / 255.0
        img = np.transpose(img, (2, 0, 1))
        return img, img_path

train_img_dir = os.path.join(DATA_ROOT_ACTIVE, "train", "images")
train_ds = YoloFolderDataset(train_img_dir, augment=custom_augment, limit=64)
train_loader = DataLoader(train_ds, batch_size=4, shuffle=True)

batch, paths = next(iter(train_loader))

fig, axes = plt.subplots(1, 4, figsize=(12, 4))
for i in range(4):
    img = batch[i].permute(1, 2, 0).cpu().numpy()
    axes[i].imshow(img)
    axes[i].axis("off")
    axes[i].set_title(os.path.basename(paths[i]))
plt.tight_layout()

## 5. Khoi tao model RT-DETR-L va optimizer
Tao model va san sang cho buoc train.

In [ ]:
model = RTDETR(MODEL_NAME)
print("Model loaded:", MODEL_NAME)

## 6. Chay thu mot vong train ngan voi augmentation
Dung so epoch nho de kiem tra pipeline; sau do tang len EPOCHS khi chay that.

In [ ]:
results = model.train(
    data=f"{DATA_ROOT_ACTIVE}/data.yaml",
    epochs=100,
    imgsz=640,
    batch=128,
    patience=30,
    optimizer="AdamW",
    lr0=1e-4,
    device=0,
    workers=4,
    project=PROJECT_DIR,
    name=RUN_NAME,
    verbose=True,
    plots=False,

    # Giu cac setting giong YOLO de so sanh cong bang
    auto_augment=None,
    close_mosaic=0,

    hsv_h=0.0,
    hsv_s=0.0,
    hsv_v=0.0,

    degrees=0.0,
    translate=0.0,
    scale=0.0,
    shear=0.0,
    perspective=0.0,

    fliplr=0.0,
    flipud=0.0,

    mosaic=0.0,
    mixup=0.0,
    copy_paste=0.0,
    erasing=0.0,
)
print("Train done:", results)

## 7. Danh gia nhanh tren batch validation
Kiem tra nhanh mAP va loss tren validation.

In [ ]:
BEST_WEIGHTS = f"/content/runs/detect/{PROJECT_DIR}/{RUN_NAME}/weights/best.pt"

assert os.path.exists(BEST_WEIGHTS), f"Khong thay {BEST_WEIGHTS}"

model = RTDETR(BEST_WEIGHTS)

last_metrics = model.val(
    data=f"{DATA_ROOT_ACTIVE}/data.yaml",
    split="val",
    verbose=False,
    plots=False,
)

print(f"mAP@50    : {last_metrics.box.map50:.4f}")
print(f"mAP@50-95 : {last_metrics.box.map:.4f}")

## 8. Luu checkpoint va log ket qua
Luu thong tin ket qua va copy run ra Drive neu can.

In [ ]:
import json
import shutil

run_dir = f"/content/runs/detect/{PROJECT_DIR}/{RUN_NAME}"
metrics_path = os.path.join(run_dir, "metrics.json")

if "last_metrics" in globals():
    with open(metrics_path, "w") as f:
        json.dump(
            {
                "map50": float(last_metrics.box.map50),
                "map50_95": float(last_metrics.box.map),
            },
            f,
            indent=2,
        )
    print("Da ghi metrics:", metrics_path)

# Copy ve Drive neu da mount
if os.path.exists("/content/drive"):
    dst = f"/content/drive/MyDrive/DAT/{RUN_NAME}"
    if os.path.exists(dst):
        shutil.rmtree(dst)
    shutil.copytree(run_dir, dst)
    print("Da copy run ve Drive:", dst)

## 9. Push HuggingFace
Day weights va log len HuggingFace (giong ben YOLO).

In [ ]:
from huggingface_hub import login, HfApi

login(token="YOUR_HF_TOKEN")  # dan token WRITE

api = HfApi()
repo_id = "Thanhdat3010/rtdetr-l-COD10K-Baseline"  # doi neu can

api.create_repo(repo_id=repo_id, exist_ok=True, repo_type="model")

run_dir = f"/content/runs/detect/{PROJECT_DIR}/{RUN_NAME}"
api.upload_folder(
    folder_path=run_dir,
    repo_id=repo_id,
    repo_type="model"
)

print("Push HuggingFace xong:", repo_id)